## Установка библиотек

Делал локально на личном ПК.

Здесь загрузка сделана через `pandas`, а не через `corus`, потому что в Python 3.13 в vs code windows у `corus` иногда падает импорт из-за зависимости от `cgi`.


In [2]:
# !pip install -q razdel pymorphy3 gensim navec scikit-learn pandas numpy matplotlib seaborn


## Импорты и общие настройки


In [1]:
import gc
import os
import re
import random
import warnings
from functools import lru_cache
from itertools import islice

import numpy as np
import pandas as pd

from razdel import tokenize
import pymorphy3

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report

from gensim.models import Word2Vec
from navec import Navec

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

## Загрузка датасета

Корпус `lenta-ru-news`.
Здесь я использую прямую загрузку `csv.gz` через `pandas` и беру семпл 100к.

In [2]:
DATA_PATH = "lenta-ru-news.csv.gz"
DATA_URL = "https://github.com/yutkin/Lenta.Ru-News-Dataset/releases/download/v1.0/lenta-ru-news.csv.gz"

if not os.path.exists(DATA_PATH):
    import urllib.request
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)

N_SAMPLES = 100_000

df = pd.read_csv(
    DATA_PATH,
    compression="gzip",
    usecols=["title", "text", "topic"],
)

if N_SAMPLES is not None and N_SAMPLES < len(df):
    df = df.sample(n=N_SAMPLES, random_state=RANDOM_STATE)

df = df.reset_index(drop=True)

print(df.shape)
df.head(3)


(100000, 3)


,title,text,topic
0,EgyptAir объявила о подорожании билетов,Египетский перевозчик EgyptAir сообщил о возмо...,Путешествия
1,Глава Красногорского района Подмосковья ушел в...,Глава Красногорского района Московской области...,Россия
2,Милонов предложил запретить россиянам сидеть в...,Депутат Виталий Милонов внес в Госдуму законоп...,Россия


## Первичная очистка и сбор текстового поля

Используем `title + text`, потому что заголовок часто несёт сильный тематический сигнал, а основной текст добавляет контекст.


In [3]:
df = df.dropna(subset=["title", "text", "topic"]).copy()

df["topic"] = (
    df["topic"]
    .astype(str)
    .str.strip()
    .str.lower()
)

df["text_full"] = (
    df["title"].astype(str).str.strip() + " " + df["text"].astype(str).str.strip()
)

df = df[df["text_full"].str.len() > 20].copy()

print(df.shape)
df[["title", "topic", "text_full"]].head(3)


(99976, 4)


,title,topic,text_full
0,EgyptAir объявила о подорожании билетов,путешествия,EgyptAir объявила о подорожании билетов Египет...
1,Глава Красногорского района Подмосковья ушел в...,россия,Глава Красногорского района Подмосковья ушел в...
2,Милонов предложил запретить россиянам сидеть в...,россия,Милонов предложил запретить россиянам сидеть в...


## Фильтрация слишком редких классов

Для стратификации и более устойчивого обучения убираем темы, которые встречаются слишком редко в выбранной подвыборке.

- стратификация `60/20/20` может ломаться на слишком редких классах;
- метрики на классах с 1-2 объектами в одной из выборок становятся шумными и плохо интерпретируются.


In [4]:
MIN_TOPIC_FREQ = 30

topic_counts = df["topic"].value_counts()
valid_topics = topic_counts[topic_counts >= MIN_TOPIC_FREQ].index
df = df[df["topic"].isin(valid_topics)].copy()

print("Осталось объектов:", len(df))
print("Осталось тем:", df["topic"].nunique())
df["topic"].value_counts().head(15)


Осталось объектов: 99956
Осталось тем: 17


topic
россия               21870
мир                  18494
экономика            10737
спорт                 8632
культура              7337
наука и техника       7129
бывший ссср           7100
интернет и сми        6181
из жизни              3718
дом                   2891
силовые структуры     2661
ценности              1079
бизнес                 967
путешествия            855
69-я параллель         178
Name: count, dtype: int64

## Предобработка текста

Здесь используется следующий пайплайн:

- приведение к нижнему регистру;
- замена URL и чисел на специальные токены;
- токенизация через `razdel`;
- лемматизация русских слов через `pymorphy3`;
- удаление очень коротких токенов и лишних символов.

Я не стал делать слишком агрессивную очистку и не удалял все стоп-слова: для тематической классификации часть служебных слов и устойчивых биграмм может быть полезна, а сильная очистка иногда ухудшает качество.
При этом уже сейчас решил отсеять слова с длиной менее 2, так как скорее всего они малозначимы. Решил также сохранить если есть специальные маркеры.


In [5]:
morph = pymorphy3.MorphAnalyzer()

URL_RE = re.compile(r"(https?://\S+|www\.\S+)", re.IGNORECASE)
NUM_RE = re.compile(r"\b\d+([.,]\d+)?\b")
LAT_CYR_WORD_RE = re.compile(r"^[a-zа-яё]+$", re.IGNORECASE)

@lru_cache(maxsize=100_000)
def lemma_token(token: str) -> str:
    if LAT_CYR_WORD_RE.fullmatch(token):
        return morph.parse(token)[0].normal_form
    return token

def preprocess_text(text: str) -> str:
    text = text.lower()
    text = URL_RE.sub(" __url__ ", text)
    text = NUM_RE.sub(" __num__ ", text)
    text = re.sub(r"[^a-zа-яё\s_]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    tokens = []
    for tok in tokenize(text):
        token = tok.text.strip()
        if not token:
            continue
        lemma = lemma_token(token)
        if lemma.startswith("__") and lemma.endswith("__"):
            tokens.append(lemma)
            continue
        if LAT_CYR_WORD_RE.fullmatch(lemma) and len(lemma) >= 2:
            tokens.append(lemma)

    return " ".join(tokens)


In [6]:
%%time

df["text_prepared"] = df["text_full"].astype(str).apply(preprocess_text)
df = df[df["text_prepared"].str.len() > 0].copy()

df[["text_full", "text_prepared"]].head(3)


CPU times: total: 1min 36s
Wall time: 1min 36s


,text_full,text_prepared
0,EgyptAir объявила о подорожании билетов Египет...,egyptair объявить подорожание билет египетский...
1,Глава Красногорского района Подмосковья ушел в...,глава красногорский район подмосковье уйти отс...
2,Милонов предложил запретить россиянам сидеть в...,милон предложить запретить россиянин сидеть со...


## Кодировка таргета и разбиение `60/20/20` со стратификацией


In [7]:
label_encoder = LabelEncoder()
df["target"] = label_encoder.fit_transform(df["topic"])

X = df["text_prepared"].values
y = df["target"].values

X_train, X_tmp, y_train, y_tmp = train_test_split(
    X,
    y,
    test_size=0.4,
    random_state=RANDOM_STATE,
    stratify=y,
)

X_valid, X_test, y_valid, y_test = train_test_split(
    X_tmp,
    y_tmp,
    test_size=0.5,
    random_state=RANDOM_STATE,
    stratify=y_tmp,
)

print(f"train: {len(X_train)} ({len(X_train)/len(X):.1%})")
print(f"valid: {len(X_valid)} ({len(X_valid)/len(X):.1%})")
print(f"test : {len(X_test)} ({len(X_test)/len(X):.1%})")


train: 59973 (60.0%)
valid: 19991 (20.0%)
test : 19992 (20.0%)


In [8]:
def show_split_distribution(name, y_part):
    dist = pd.Series(y_part).value_counts(normalize=True).sort_index()
    return pd.DataFrame({name: dist})

split_dist = (
    show_split_distribution("train", y_train)
    .join(show_split_distribution("valid", y_valid), how="outer")
    .join(show_split_distribution("test", y_test), how="outer")
    .fillna(0)
)

split_dist.head()


,train,valid,test
0,0.001784,0.001801,0.001751
1,0.009671,0.009704,0.009654
2,0.071032,0.071032,0.071028
3,0.028930,0.028913,0.028912
4,0.037200,0.037167,0.037215


## Вспомогательные функции для оценки качества

Основная метрика ниже — `accuracy` (так как тематик много, нет концентрации именно в одной, интересует доля правильных от всего числа). 

Дополнительно смотрим `macro F1`, так как задача многоклассовая и темы распределены неравномерно


In [9]:
def evaluate_predictions(y_true, y_pred, model_name: str, split_name: str) -> dict:
    return {
        "model": model_name,
        "split": split_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
    }

def fit_eval_pipeline(pipe, X_tr, y_tr, X_va, y_va, model_name: str):
    pipe.fit(X_tr, y_tr)
    valid_pred = pipe.predict(X_va)
    metrics = evaluate_predictions(y_va, valid_pred, model_name=model_name, split_name="valid")
    return pipe, metrics


## 1) `CountVectorizer` + `LogisticRegression`


In [10]:
count_pipe = Pipeline(
    steps=[
        (
            "vect",
            CountVectorizer(
                token_pattern=r"(?u)\b[\wёЁ]+\b",
                ngram_range=(1, 2),
                min_df=3,
                max_df=0.8,
                max_features=100_000,
            ),
        ),
        (
            "clf",
            LogisticRegression(
                solver="saga",
                max_iter=1000,
                C=2.0,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

count_pipe, count_valid_metrics = fit_eval_pipeline(
    count_pipe, X_train, y_train, X_valid, y_valid, "CountVectorizer + LogisticRegression"
)

pd.DataFrame([count_valid_metrics])


,model,split,accuracy,macro_f1
0,CountVectorizer + LogisticRegression,valid,0.819419,0.689672


## 2) `TfidfVectorizer` + `LogisticRegression`


In [11]:
tfidf_pipe = Pipeline(
    steps=[
        (
            "vect",
            TfidfVectorizer(
                token_pattern=r"(?u)\b[\wёЁ]+\b",
                ngram_range=(1, 2),
                min_df=3,
                max_df=0.8,
                max_features=100_000,
                sublinear_tf=True,
            ),
        ),
        (
            "clf",
            LogisticRegression(
                solver="saga",
                max_iter=1000,
                C=2.0,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

tfidf_pipe, tfidf_valid_metrics = fit_eval_pipeline(
    tfidf_pipe, X_train, y_train, X_valid, y_valid, "TF-IDF + LogisticRegression"
)

pd.DataFrame([tfidf_valid_metrics])


,model,split,accuracy,macro_f1
0,TF-IDF + LogisticRegression,valid,0.82127,0.636808


## Краткий промежуточный вывод

На валидации по основной метрике `accuracy` немного лучше показала себя модель **`TF-IDF + LogisticRegression`**: `0.8213` против `0.8194` у `CountVectorizer + LogisticRegression`. При этом по `macro F1` лучше оказался **`CountVectorizer + LogisticRegression`**: `0.6897` против `0.6368`, то есть bag-of-words чуть ровнее отработал по менее частотным темам. Это объяснимо: `TF-IDF` лучше подавляет слишком частотные слова и подчеркивает редкие тематические маркеры, а объединение `title + text`, лемматизация и использование биграмм усиливают сигнал темы и помогают учитывать устойчивые словосочетания.


## Подбор гиперпараметров на кросс-валидации

Чтобы не делать слишком тяжёлый перебор, используем `RandomizedSearchCV` с небольшим числом конфигураций.

Ниже я перебираю:
- тип векторизации (`CountVectorizer` / `TfidfVectorizer`);
- диапазон `ngram_range`;
- пороги `min_df`, `max_df`;
- размер регуляризации `C`;
- `class_weight`.


In [13]:
search_pipe = Pipeline(
    steps=[
        ("vect", TfidfVectorizer()),
        (
            "clf",
            LogisticRegression(
                solver="saga",
                max_iter=1000,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

param_distributions = [
    {
        "vect": [
            CountVectorizer(token_pattern=r"(?u)\b[\wёЁ]+\b", max_features=100_000),
        ],
        "vect__ngram_range": [(1, 1), (1, 2)],
        "vect__min_df": [3, 10, 25],
        "vect__max_df": [0.7, 0.8, 0.9],
        "clf__C": [0.5, 1.0, 2.0, 4.0],
        "clf__class_weight": [None, "balanced"],
    },
    {
        "vect": [
            TfidfVectorizer(
                token_pattern=r"(?u)\b[\wёЁ]+\b",
                max_features=100_000,
            ),
        ],
        "vect__ngram_range": [(1, 1), (1, 2)],
        "vect__min_df": [3, 10, 25],
        "vect__max_df": [0.7, 0.8, 0.9],
        "vect__sublinear_tf": [True, False],
        "clf__C": [0.5, 1.0, 2.0, 4.0],
        "clf__class_weight": [None, "balanced"],
    },
]

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

search = RandomizedSearchCV(
    estimator=search_pipe,
    param_distributions=param_distributions,
    n_iter=12,
    scoring="accuracy",
    n_jobs=1,
    cv=cv,
    verbose=1,
    random_state=RANDOM_STATE,
)

search.fit(X_train, y_train)

search.best_params_, search.best_score_


Fitting 3 folds for each of 12 candidates, totalling 36 fits


({'vect__ngram_range': (1, 2),
  'vect__min_df': 10,
  'vect__max_df': 0.7,
  'vect': CountVectorizer(max_features=100000, token_pattern='(?u)\\b[\\wёЁ]+\\b'),
  'clf__class_weight': None,
  'clf__C': 2.0},
 np.float64(0.8101312257182399))

In [14]:
best_cv_model = search.best_estimator_
best_valid_pred = best_cv_model.predict(X_valid)

best_cv_valid_metrics = evaluate_predictions(
    y_valid,
    best_valid_pred,
    model_name="Best CV sparse model",
    split_name="valid",
)

pd.DataFrame([best_cv_valid_metrics])


,model,split,accuracy,macro_f1
0,Best CV sparse model,valid,0.818818,0.688552


## Почему я остановился на таком пайплайне

В качестве текстового представления я использовал объединение `title` и `text`, поскольку заголовок часто содержит сильный тематический сигнал, а основной текст добавляет контекст.  
На этапе предобработки были выполнены приведение к нижнему регистру, замена URL и чисел на специальные токены, токенизация и лемматизация. Для русского языка лемматизация особенно полезна, так как уменьшает размер словаря и объединяет разные словоформы.  
Далее были протестированы два sparse-подхода: `CountVectorizer` и `TfidfVectorizer` в сочетании с `LogisticRegression`. Затем параметры векторизации и модели подбирались по кросс-валидации на обучающей выборке.  
Такой пайплайн даёт сильный и интерпретируемый baseline для задачи тематической классификации новостей и при этом остаётся вычислительно доступным.


## Подготовка токенов для `word2vec`

Для обучения своих эмбеддингов берём **только train**, чтобы не подглядывать в валидацию и тест.


In [15]:
train_sentences = [text.split() for text in X_train]
valid_sentences = [text.split() for text in X_valid]
test_sentences  = [text.split() for text in X_test]

print(train_sentences[0][:20])


['госдума', 'начало', 'реформа', 'железнодорожный', 'транспорт', 'госдума', 'на', 'сегодняшний', 'заседание', 'принять', 'первый', 'чтение', 'проект', 'федеральный', 'закон', 'железнодорожный', 'транспорт', 'рф', 'сообщать', 'ак']


## Обучение своих `word2vec`-эмбеддингов (`gensim`)

Ниже выбрана разумная стартовая конфигурация для новостного корпуса:

- `vector_size=200` — достаточно ёмкое представление слов без слишком большого роста времени обучения;
- `window=5` — локальный контекст подходит для новостных текстов;
- `min_count=5` — отсечение крайне редких слов и шумовых токенов;
- `sg=1` — Skip-gram обычно лучше захватывает семантику редких слов;
- `negative=10` — стандартный и устойчивый вариант negative sampling;
- `epochs=10` — несколько проходов по корпусу позволяют получить более устойчивые вектора.

Важно, что для промежуточного сравнения модель обучается **только на train**, чтобы не было утечки информации из validation и test.


In [16]:
w2v_model = Word2Vec(
    sentences=train_sentences,
    vector_size=200,
    window=5,
    min_count=5,
    workers=max(1, os.cpu_count() // 2),
    sg=1,
    negative=10,
    sample=1e-4,
    epochs=10,
    seed=RANDOM_STATE,
)


## Внутренняя (intrinsic) оценка эмбеддингов

Проверим две стандартные вещи из `gensim`:
- `most_similar` — ближайшие слова;
- `doesnt_match` — лишнее слово в наборе.

Если какое-то слово не попало в словарь, можно заменить его на другое частотное.


In [17]:
def safe_most_similar(model, word, topn=10):
    if word in model.wv:
        return model.wv.most_similar(word, topn=topn)
    return f"Слово '{word}' не найдено в словаре"

def safe_doesnt_match(model, words):
    words_in_vocab = [w for w in words if w in model.wv]
    if len(words_in_vocab) >= 3:
        return model.wv.doesnt_match(words_in_vocab)
    return f"Недостаточно слов из словаря: {words_in_vocab}"

safe_most_similar(w2v_model, "россия", topn=10)


[('рф', 0.7298049330711365),
 ('российский', 0.6834427118301392),
 ('страна', 0.6585671901702881),
 ('украина', 0.6160023808479309),
 ('тулина', 0.6035524010658264),
 ('видеомост', 0.5839394927024841),
 ('москва', 0.5823220610618591),
 ('ельченко', 0.5770885348320007),
 ('федоткина', 0.5740891695022583),
 ('фарит', 0.5727470517158508)]

In [18]:
safe_most_similar(w2v_model, "москва", topn=10)


[('столичный', 0.6789741516113281),
 ('столица', 0.674426257610321),
 ('московский', 0.6336702704429626),
 ('переров', 0.6207392811775208),
 ('марьино', 0.617273211479187),
 ('тихорецкий', 0.6036473512649536),
 ('богородский', 0.6026619076728821),
 ('кузьминский', 0.5947118401527405),
 ('калуга', 0.5923334360122681),
 ('лиговский', 0.5912157893180847)]

In [19]:
safe_doesnt_match(w2v_model, ["январь", "февраль", "март", "апрель", "собака"])


'собака'

Проверка `word2vec` показала, что эмбеддинги обучились осмысленно. У слова `россия` среди ближайших соседей оказались `рф`, `российский`, `страна`, `украина`, `москва`, то есть модель действительно улавливает географический и политический контекст. Для слова `москва` среди соседей появились `столичный`, `столица`, `московский` и названия районов, что тоже выглядит логично. В задаче `doesnt_match` лишним словом корректно выбралась `собака`, значит даже простая intrinsic-проверка показывает, что векторное пространство частично схватывает семантическую близость слов.


## Загрузка предобученных эмбеддингов `Navec`

Использую `Navec`, потому что это компактные русскоязычные эмбеддинги, их удобно быстро скачать и применить в домашней работе.


In [20]:
NAVEC_PATH = "navec_news_v1_1B_250K_300d_100q.tar"
NAVEC_URL = "https://storage.yandexcloud.net/natasha-navec/packs/navec_news_v1_1B_250K_300d_100q.tar"

if not os.path.exists(NAVEC_PATH):
    import urllib.request
    urllib.request.urlretrieve(NAVEC_URL, NAVEC_PATH)

navec = Navec.load(NAVEC_PATH)
navec["москва"][:10]


array([-0.40779433,  0.0541666 ,  0.33955213,  0.41585353, -0.12587811,
        0.22305727, -0.08312037, -0.16948785,  0.3331682 , -0.28451073],
      dtype=float32)

## Усреднение эмбеддингов по документу

Для классификации новости каждую новость превращаем в один вектор как среднее по векторам её токенов.  
Это простой, но понятный baseline для сравнения своих и предобученных эмбеддингов.


In [21]:
def get_w2v_vector(token: str):
    return w2v_model.wv[token] if token in w2v_model.wv else None

def get_navec_vector(token: str):
    try:
        return navec[token]
    except KeyError:
        return None

def text_to_mean_vector(tokens, vector_getter, dim):
    vectors = [vector_getter(tok) for tok in tokens]
    vectors = [vec for vec in vectors if vec is not None]
    if len(vectors) == 0:
        return np.zeros(dim, dtype=np.float32)
    return np.mean(vectors, axis=0).astype(np.float32)

def build_matrix(sentences, vector_getter, dim):
    return np.vstack([text_to_mean_vector(tokens, vector_getter, dim) for tokens in sentences])


In [22]:
W2V_DIM = w2v_model.wv.vector_size
NAVEC_DIM = len(navec["москва"])

X_train_w2v = build_matrix(train_sentences, get_w2v_vector, W2V_DIM)
X_valid_w2v = build_matrix(valid_sentences, get_w2v_vector, W2V_DIM)
X_test_w2v  = build_matrix(test_sentences,  get_w2v_vector, W2V_DIM)

X_train_navec = build_matrix(train_sentences, get_navec_vector, NAVEC_DIM)
X_valid_navec = build_matrix(valid_sentences, get_navec_vector, NAVEC_DIM)
X_test_navec  = build_matrix(test_sentences,  get_navec_vector, NAVEC_DIM)

X_train_w2v.shape, X_train_navec.shape


((59973, 200), (59973, 300))

## Покрытие токенов эмбеддингами

Небольшая дополнительная проверка, которая улучшает качество отчёта: посмотрим, какая доля токенов из выборок покрывается своими `word2vec` и предобученными `Navec`.


In [23]:
def embedding_coverage(sentences, vector_getter):
    total = 0
    covered = 0
    for sent in sentences:
        for tok in sent:
            total += 1
            if vector_getter(tok) is not None:
                covered += 1
    return covered / total if total else 0.0

coverage_df = pd.DataFrame(
    {
        "embedding": ["own_w2v", "navec"],
        "train_coverage": [
            embedding_coverage(train_sentences, get_w2v_vector),
            embedding_coverage(train_sentences, get_navec_vector),
        ],
        "valid_coverage": [
            embedding_coverage(valid_sentences, get_w2v_vector),
            embedding_coverage(valid_sentences, get_navec_vector),
        ],
    }
)

coverage_df


,embedding,train_coverage,valid_coverage
0,own_w2v,0.982819,0.977417
1,navec,0.965643,0.966060


## `LogisticRegression` на своих `word2vec`-эмбеддингах


In [24]:
w2v_clf = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        (
            "clf",
            LogisticRegression(
                max_iter=1000,
                C=2.0,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

w2v_clf.fit(X_train_w2v, y_train)
w2v_valid_pred = w2v_clf.predict(X_valid_w2v)

w2v_valid_metrics = evaluate_predictions(
    y_valid,
    w2v_valid_pred,
    model_name="Own Word2Vec + LogisticRegression",
    split_name="valid",
)

pd.DataFrame([w2v_valid_metrics])


,model,split,accuracy,macro_f1
0,Own Word2Vec + LogisticRegression,valid,0.793757,0.646461


## `LogisticRegression` на предобученных `Navec`-эмбеддингах


In [25]:
navec_clf = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        (
            "clf",
            LogisticRegression(
                max_iter=1000,
                C=2.0,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

navec_clf.fit(X_train_navec, y_train)
navec_valid_pred = navec_clf.predict(X_valid_navec)

navec_valid_metrics = evaluate_predictions(
    y_valid,
    navec_valid_pred,
    model_name="Navec + LogisticRegression",
    split_name="valid",
)

pd.DataFrame([navec_valid_metrics])


,model,split,accuracy,macro_f1
0,Navec + LogisticRegression,valid,0.785003,0.63966


## Сравнение моделей на валидации


In [26]:
valid_results = pd.DataFrame(
    [
        count_valid_metrics,
        tfidf_valid_metrics,
        best_cv_valid_metrics,
        w2v_valid_metrics,
        navec_valid_metrics,
    ]
).sort_values(["accuracy", "macro_f1"], ascending=False)

valid_results


,model,split,accuracy,macro_f1
1,TF-IDF + LogisticRegression,valid,0.821270,0.636808
0,CountVectorizer + LogisticRegression,valid,0.819419,0.689672
2,Best CV sparse model,valid,0.818818,0.688552
3,Own Word2Vec + LogisticRegression,valid,0.793757,0.646461
4,Navec + LogisticRegression,valid,0.785003,0.639660


На валидации по `accuracy` выиграла модель **`TF-IDF + LogisticRegression`** (`0.8213`), но по `macro F1` сильнее оказался **`CountVectorizer + LogisticRegression`** (`0.6897`), поэтому sparse-подходы в целом дали лучший результат. Обе разреженные модели заметно обошли классификацию на усреднённых эмбеддингах: собственный `Word2Vec` дал `accuracy = 0.7938`, а `Navec` — `0.7850`. Это ожидаемо, поскольку для тематической классификации новостей особенно важны редкие маркерные слова и биграммы, а простое усреднение эмбеддингов теряет часть структуры текста и ослабляет сигнал отдельных терминов. Предобученные эмбеддинги `Navec` здесь оказались немного слабее собственных `word2vec`, вероятно потому, что модель, обученная на train-подвыборке, лучше подстроилась под лексику и стиль корпуса `Lenta.ru`; это косвенно подтверждается и чуть более высоким покрытием токенов у собственного `Word2Vec`.

## Финальное сравнение на тестовой выборке

Ниже я переобучаю каждую конфигурацию на `train + valid`, а затем один раз оцениваю на `test`.


In [27]:
X_trainval = np.concatenate([X_train, X_valid])
y_trainval = np.concatenate([y_train, y_valid])

trainval_sentences = [text.split() for text in X_trainval]


In [28]:
# 1. CountVectorizer + LR
count_final = Pipeline(
    steps=[
        (
            "vect",
            CountVectorizer(
                token_pattern=r"(?u)\b[\wёЁ]+\b",
                ngram_range=(1, 2),
                min_df=3,
                max_df=0.8,
                max_features=100_000,
            ),
        ),
        (
            "clf",
            LogisticRegression(
                solver="saga",
                max_iter=1000,
                C=2.0,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)
count_final.fit(X_trainval, y_trainval)
count_test_pred = count_final.predict(X_test)
count_test_metrics = evaluate_predictions(y_test, count_test_pred, "CountVectorizer + LogisticRegression", "test")


In [29]:
# 2. TF-IDF + LR
tfidf_final = Pipeline(
    steps=[
        (
            "vect",
            TfidfVectorizer(
                token_pattern=r"(?u)\b[\wёЁ]+\b",
                ngram_range=(1, 2),
                min_df=3,
                max_df=0.8,
                max_features=100_000,
                sublinear_tf=True,
            ),
        ),
        (
            "clf",
            LogisticRegression(
                solver="saga",
                max_iter=1000,
                C=2.0,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)
tfidf_final.fit(X_trainval, y_trainval)
tfidf_test_pred = tfidf_final.predict(X_test)
tfidf_test_metrics = evaluate_predictions(y_test, tfidf_test_pred, "TF-IDF + LogisticRegression", "test")


In [30]:
# 3. Best CV sparse model
best_cv_final = search.best_estimator_
best_cv_final.fit(X_trainval, y_trainval)
best_cv_test_pred = best_cv_final.predict(X_test)
best_cv_test_metrics = evaluate_predictions(y_test, best_cv_test_pred, "Best CV sparse model", "test")


In [31]:
# 4. Own Word2Vec + LR (переобучаем w2v на train+valid)
w2v_final = Word2Vec(
    sentences=trainval_sentences,
    vector_size=200,
    window=5,
    min_count=5,
    workers=max(1, os.cpu_count() // 2),
    sg=1,
    negative=10,
    sample=1e-4,
    epochs=10,
    seed=RANDOM_STATE,
)

def get_w2v_final_vector(token: str):
    return w2v_final.wv[token] if token in w2v_final.wv else None

X_trainval_w2v = build_matrix(trainval_sentences, get_w2v_final_vector, w2v_final.wv.vector_size)
X_test_w2v_final = build_matrix(test_sentences, get_w2v_final_vector, w2v_final.wv.vector_size)

w2v_final_clf = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=2000, C=2.0, random_state=RANDOM_STATE)),
    ]
)
w2v_final_clf.fit(X_trainval_w2v, y_trainval)
w2v_test_pred = w2v_final_clf.predict(X_test_w2v_final)
w2v_test_metrics = evaluate_predictions(y_test, w2v_test_pred, "Own Word2Vec + LogisticRegression", "test")


In [32]:
# 5. Navec + LR
X_trainval_navec = build_matrix(trainval_sentences, get_navec_vector, NAVEC_DIM)
X_test_navec_final = build_matrix(test_sentences, get_navec_vector, NAVEC_DIM)

navec_final_clf = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, C=2.0, random_state=RANDOM_STATE)),
    ]
)
navec_final_clf.fit(X_trainval_navec, y_trainval)
navec_test_pred = navec_final_clf.predict(X_test_navec_final)
navec_test_metrics = evaluate_predictions(y_test, navec_test_pred, "Navec + LogisticRegression", "test")


In [35]:
test_results = pd.DataFrame(
    [
        count_test_metrics,
        tfidf_test_metrics,
        best_cv_test_metrics,
        w2v_test_metrics,
        navec_test_metrics,
    ]
).sort_values(["accuracy", "macro_f1"], ascending=False)

test_results


,model,split,accuracy,macro_f1
1,TF-IDF + LogisticRegression,test,0.828681,0.664617
0,CountVectorizer + LogisticRegression,test,0.828481,0.694743
2,Best CV sparse model,test,0.827931,0.692597
3,Own Word2Vec + LogisticRegression,test,0.798970,0.642204
4,Navec + LogisticRegression,test,0.784614,0.638479


## Итоговый вывод

На тестовой выборке по основной метрике `accuracy` лучшей оказалась модель **`TF-IDF + LogisticRegression`**, показавшая `accuracy = 0.8287` и `macro F1 = 0.6646`. При этом по `macro F1` лучшим sparse-подходом оказался **`CountVectorizer + LogisticRegression`**: `0.6947` при `accuracy = 0.8285`, то есть обе разреженные модели выступили очень близко, но `CountVectorizer` чуть ровнее отработал по менее частотным классам. Модели на усреднённых эмбеддингах оказались слабее: **`Own Word2Vec + LogisticRegression`** дал `accuracy = 0.7990` и `macro F1 = 0.6422`, а **`Navec + LogisticRegression`** — `accuracy = 0.7846` и `macro F1 = 0.6385`, поскольку усреднение векторов частично теряет порядок слов, локальный контекст и силу редких тематических признаков. Предобученные эмбеддинги `Navec` в этой работе оказались хуже собственных `word2vec`, вероятно потому, что собственная модель обучалась на том же новостном домене и лучше уловила лексику и связи внутри корпуса `Lenta.ru`.


In [37]:
# Необязательно: подробный отчёт по лучшей модели на тесте
best_model_name = test_results.iloc[0]["model"]
print("Лучшая модель на тесте:", best_model_name)

pred_map = {
    "CountVectorizer + LogisticRegression": count_test_pred,
    "TF-IDF + LogisticRegression": tfidf_test_pred,
    "Best CV sparse model": best_cv_test_pred,
    "Own Word2Vec + LogisticRegression": w2v_test_pred,
    "Navec + LogisticRegression": navec_test_pred,
}

best_test_pred = pred_map[best_model_name]
print(classification_report(y_test, best_test_pred, target_names=label_encoder.classes_))


Лучшая модель на тесте: TF-IDF + LogisticRegression
                   precision    recall  f1-score   support

   69-я параллель       0.86      0.17      0.29        35
           бизнес       0.85      0.26      0.40       193
      бывший ссср       0.85      0.85      0.85      1420
              дом       0.89      0.80      0.85       578
         из жизни       0.72      0.57      0.64       744
   интернет и сми       0.78      0.70      0.74      1236
             крым       1.00      0.12      0.22        16
     культпросвет       0.00      0.00      0.00         9
         культура       0.87      0.89      0.88      1468
              мир       0.81      0.85      0.83      3699
  наука и техника       0.85      0.85      0.85      1426
      путешествия       0.86      0.54      0.66       171
           россия       0.79      0.86      0.83      4374
силовые структуры       0.78      0.53      0.63       532
            спорт       0.96      0.96      0.96      1727
   